Protocolo vigente: una trayectoria de 100 generaciones de carácter evolutivo (algoritmo evolutivo poblacional), sin diseño base. Luz 20 m, alturas y secciones variables. Objetivos: masa mínima, utilización promedio máxima y desplazamiento Z mínimo. Criterio principal L/700; transferencia cronológica complementaria. Los resultados previos deben regenerarse tras el entrenamiento.


# 03 — Seleccionar alternativas con un presupuesto de consultas

**Pregunta:** ¿cómo usar las predicciones para encontrar buenas alternativas consultando pocos diseños?

El catálogo contiene diseños registrados. Al inicio se conocen sus entradas y predicciones; las respuestas reales se revelan solo al consultar cada candidato. Una consulta consume presupuesto aunque el diseño no mejore el conjunto encontrado. No se generan geometrías nuevas.


In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Image, Markdown, display

from truss_research.evaluacion.presentacion import notebook_context

context = notebook_context()
STUDY_DIR, REPORT_DIR, CONFIG = context.study_dir, context.report_dir, context.config
DATA_DIR, STATE, RECORDS = CONFIG.data_dir, context.state, context.records
display(
    Markdown(
        f"**Estudio:** `{STUDY_DIR.name}` · **etiqueta:** `{CONFIG.label}` · **informe:** `{REPORT_DIR.name}`"
    )
)
if CONFIG.label == "comprobacion":
    display(
        Markdown(
            "**Comprobación técnica breve: sus métricas no constituyen resultados de tesis.**"
        )
    )

## 1. Experimento común

Se optimizan simultáneamente **menor masa, menor desplazamiento y mayor utilización promedio**. Se usa `F = (masa, -utilización, desplazamiento)` para expresar los tres como minimización, conservando utilización ≤ 1 y desplazamiento ≤ 20/700 m. Pareto conserva alternativas sin otra que mejore al menos un objetivo sin empeorar los demás. El hipervolumen mide el volumen dominado en este espacio normalizado de tres objetivos; no es volumen físico.

Se comparan azar, ranking, greedy, NSGA-II y NSGA-II asistido. Se añaden Double DQN, Dueling Double DQN y PPO, cada uno con y sin predicciones. Comparten catálogo, inicio y presupuesto; la primera consulta cuenta. El protocolo principal usa 32 candidatos y presupuestos 8/16/24.

Gymnasium organiza episodios; TensorDict representa transiciones; TorchRL aporta replay, pérdida Double DQN, PPO categórico enmascarado y GAE. La formulación tabular y la red por candidato pertenecen al proyecto.


In [ ]:
display(
    pd.Series(
        {
            "candidatos_por_catálogo": CONFIG.catalog_size,
            "presupuestos": CONFIG.budgets,
            "semillas_de_catálogo_y_política": CONFIG.seeds,
            "catálogos_por_semilla": CONFIG.episodes_per_seed,
            "transiciones_por_política": CONFIG.rl_steps,
            "validación_cada": CONFIG.validation_interval,
            "evolución_activada": CONFIG.evolution.enabled,
            "población_evolutiva": CONFIG.evolution.population_size
            if CONFIG.evolution.enabled
            else "no ejecutada",
        }
    )
)
traces = pd.read_csv(REPORT_DIR / "selection_traces.csv")
selection = pd.read_csv(REPORT_DIR / "selection_per_seed.csv")
curves = pd.read_csv(REPORT_DIR / "learning_curves.csv")

## 2. Seguir un episodio guardado

La tabla muestra las consultas efectivamente realizadas; no vuelve a ejecutar la selección. La referencia del catálogo completo pertenece al evaluador retrospectivo y no se entrega a la política. La columna `reward` representa la contribución trapezoidal al área por consulta, sin normalización por la referencia oculta durante evaluación. `hypervolume_increment` muestra cuánto aumentó el hipervolumen; una contribución al área puede ser positiva aunque el incremento sea cero. El entrenamiento normaliza el área exclusivamente con su catálogo privado de entrenamiento.


In [ ]:
validation_trace = traces.loc[traces["split"].eq("validation")]
episode_id = validation_trace.iloc[0]["episode_id"]
example = validation_trace.loc[
    validation_trace["episode_id"].eq(episode_id)
    & validation_trace["comparison_id"].isin(
        (
            "random",
            "greedy",
            "nsga2",
            "nsga2_surrogate",
            "ensemble_mean",
            "ensemble_ucb",
            f"rl_{CONFIG.representative_seed}",
        )
    )
]
display(
    example[
        [
            "strategy",
            "query",
            "iter_id",
            "mass_kg",
            "util_ratio",
            "z_disp_m",
            "reward",
            "relative_hypervolume",
        ]
    ]
)
for strategy, group in example.groupby("strategy"):
    plt.plot(group["query"], group["relative_hypervolume"], marker="o", label=strategy)
plt.xlabel("Consultas, incluido el inicio")
plt.ylabel("Fracción del hipervolumen del catálogo")
plt.title(f"Episodio {episode_id}")
plt.legend()
plt.tight_layout()
plt.show()

## 3. Entender qué aprende RL

Double DQN asigna una puntuación a cada candidato disponible y aprende de la mejora observada. Durante entrenamiento recibe predicciones fuera de pliegue. En evaluación, predictor y política quedan congelados.

El checkpoint se selecciona con validación. La ablación conserva entradas e historial consultado, pero elimina las predicciones de su codificador. Esto permite investigar el aporte del predictor sin cambiar las demás condiciones.


Dueling separa valor del estado y ventaja por candidato. PPO aprende logits por candidato y un crítico sobre el contexto público. PPO muestrea durante entrenamiento y usa argmax en evaluación, como las otras políticas. Se comparan parámetros fijos, no algoritmos ajustados a su mejor configuración posible. Los tres algoritmos usan la misma secuencia de catálogos por semilla y comparten los episodios de evaluación.


In [ ]:
display(Image(filename=str(REPORT_DIR / "rl_learning_transitions.png")))

display(
    selection.query("split == 'validation'")[
        [
            "strategy",
            "policy_training_seed",
            "budget",
            "normalized_auc",
            "relative_hypervolume",
            "target_reached",
        ]
    ]
)

## 4. Comparar calidad durante las consultas

El AUC normalizado resume la calidad a lo largo del presupuesto: un valor mayor es mejor. El hipervolumen relativo final describe la calidad al terminar. Alcanzar el 90 % se informa junto con la fracción de episodios que lo consigue; no se ocultan los que no llegan.


In [ ]:
budgets = sorted(validation_trace["budget"].unique())
figure, axes = plt.subplots(
    1, len(budgets), figsize=(5 * len(budgets), 4), squeeze=False
)
for budget, axis in zip(budgets, axes.flat, strict=True):
    grouped = (
        validation_trace.loc[validation_trace["budget"].eq(budget)]
        .groupby(["strategy", "query"])["relative_hypervolume"]
        .mean()
    )
    for strategy, values in grouped.groupby(level="strategy"):
        axis.plot(
            values.index.get_level_values("query"), values.to_numpy(), label=strategy
        )
    axis.set(
        title=f"Presupuesto {budget}",
        xlabel="Consultas",
        ylabel="Hipervolumen relativo",
    )
    axis.legend(fontsize="small")
plt.tight_layout()
plt.show()
differences = pd.read_csv(REPORT_DIR / "paired_differences.csv")
display(
    differences.query("split == 'validation'")
    .groupby(["strategy", "reference_strategy"])["auc_difference"]
    .agg(["mean", "std", "count"])
)

## 5. Adaptación evolutiva

NSGA-II selecciona padres consultados, cruza entradas y aplica mutación. Cada propuesta se proyecta al candidato disponible más cercano; su respuesta pertenece al diseño registrado. La proyección puede alterar la propuesta: se registran distancia, colisiones y consultas efectivas.

La población principal es cuatro; quedan 4, 12 o 20 consultas posteriores a inicialización. La comprobación usa población dos. La variante asistida preselecciona entre hasta ocho propuestas con el predictor congelado; la supervivencia usa respuestas reveladas. Cada consulta cuenta aunque luego se descarte el individuo.

Esta adaptación utiliza pymoo; no reproduce HIGA, MulCT o SA-CoDE. Los registros permiten revisar el comportamiento real.


In [ ]:
evolution_trace = validation_trace.loc[
    validation_trace["strategy"].isin(("nsga2", "nsga2_surrogate"))
]
if evolution_trace.empty:
    display(Markdown("Este protocolo no incluye baselines evolutivos."))
else:
    offspring = evolution_trace.loc[evolution_trace["evolution_stage"].eq("offspring")]
    display(
        offspring.groupby("strategy").agg(
            consultas_evolutivas=("query", "count"),
            propuestas_por_consulta=("proposal_count", "mean"),
            distancia_media=("projection_distance", "mean"),
            colisiones=("projection_collision_count", "sum"),
            distancia_maxima=("projection_distance", "max"),
            recursos_aleatorios=("random_fallback", "sum"),
        )
    )
    display(
        offspring[
            [
                "strategy",
                "query",
                "iter_id",
                "proposal_web_config",
                "proposal_n_trusses",
                "proposal_lr_height_m",
                "projection_distance",
                "reward",
            ]
        ].head(8)
    )
    from truss_research.cerchas.schemas import INPUT_COLUMNS
    from truss_research.datos.procesados import load_design_split

    registered = load_design_split(DATA_DIR).validation.set_index("iter_id")
    display(
        Markdown(
            "**Entradas registradas de los diseños consultados:** comparar estos valores con las propuestas anteriores."
        )
    )
    display(registered.loc[offspring["iter_id"].head(8), list(INPUT_COLUMNS)])

## 6. Costos y magnitudes físicas

La preparación de ranking se incluye en `strategy_seconds`, junto con las elecciones. Inferencia cacheada, preparación del banco y carga de políticas se miden una vez por comparación. No se multiplican por episodio ni se comparan con FEM.

El gráfico tridimensional muestra masa, utilización promedio y desplazamiento registrados de los diseños consultados. Los tres intervienen en la búsqueda y el hipervolumen. No representa estructuras nuevas simuladas.


In [ ]:
display(pd.read_csv(REPORT_DIR / "selection_shared_costs.csv"))
display(
    selection.query("split == 'validation'")[
        [
            "strategy",
            "budget",
            "preparation_seconds",
            "decision_seconds",
            "strategy_seconds",
        ]
    ]
)
figure = plt.figure(figsize=(10, 7))
axis = figure.add_subplot(projection="3d")
for strategy, group in example.groupby("strategy"):
    axis.scatter(
        group["mass_kg"],
        group["util_ratio"],
        group["z_disp_m"],
        label=strategy,
        alpha=0.7,
    )
axis.set(
    xlabel="Masa registrada (kg)",
    ylabel="Utilización promedio",
    zlabel="Desplazamiento registrado (m)",
    title="Diseños consultados: tres objetivos simultáneos",
)
axis.legend()
plt.tight_layout()
plt.show()

## Conclusión de esta etapa

Las diferencias se comparan sobre los mismos episodios. Los catálogos comparten diseños y los recorridos no son muestras independientes. RL puede mejorar al azar y aun así ser superado por greedy; también puede resultar que añadir predicciones no mejore su política. Reducir consultas en este catálogo no demuestra ahorro temporal frente al software de origen.


## Calidad temprana y sensibilidades

AUC normalizada es la medida principal. RL se entrena con incrementos marginales ponderados equivalentes al AUC y descuento 1; las ablaciones conservan la recompensa trapezoidal, usando normalización privada solo en entrenamiento. Las referencias completas no entran en observaciones. make sensitivity evalúa catálogos ampliados con políticas congeladas y recalcula márgenes de referencia sobre trayectorias fijas; sus resultados se exportan aparte.


In [ ]:
display(pd.read_csv(REPORT_DIR / "selection_intervals.csv"))
if context.sensitivity_dir is None:
    display(
        Markdown(
            "Sensibilidades pendientes: ejecutar make sensitivity después del cierre."
        )
    )
else:
    display(Markdown(f"**Sensibilidad verificada:** `{context.sensitivity_dir.name}`"))
    display(pd.read_csv(context.sensitivity_dir / "catalog_summary.csv"))
    for figure in sorted(context.sensitivity_dir.glob("*.png")):
        display(Image(filename=str(figure)))

## Diagnósticos de robustez
Los errores extremos se conservan y se presentan por separado. La cobertura escasa y la procedencia física pendiente limitan las conclusiones; no se retocan modelos a partir de prueba.

In [ ]:
from truss_research.evaluacion.robustez import EVIDENCE_NOTES

display(Markdown("\n\n".join(EVIDENCE_NOTES.values())))
for diagnostic in (
    "error_tail_summary.csv",
    "category_coverage.csv",
    "physical_provenance.csv",
    "predictive_seed_variation.csv",
    "convergence_diagnostics.csv",
):
    diagnostic_path = REPORT_DIR / diagnostic
    if diagnostic_path.is_file():
        display(pd.read_csv(diagnostic_path).head(20))

## Costo y aprendizaje por familia RL

Las seis variantes comparten transiciones y episodios. Las actualizaciones y muestras procesadas pueden diferir; por ello se informa también aprendizaje frente a CPU y duración. Las figuras promedian semillas en hitos comunes, excluyen validación del tiempo de entrenamiento y no representan aceleración del solver. Las pérdidas de DQN y PPO no se comparan directamente.


In [ ]:
display(pd.read_csv(REPORT_DIR / "policy_training_summary.csv"))
for axis in ("transitions", "cpu", "wall"):
    display(Image(filename=str(REPORT_DIR / f"rl_learning_{axis}.png")))

## Recursos computacionales y costo de uso

Se separan predicción del catálogo, preparación y decisiones del entorno simulado. Solo las estrategias que utilizan predicciones pagan inferencia. Los catálogos consultan respuestas registradas; estos tiempos no acreditan aceleración frente a un solver estructural.

In [ ]:
from IPython.display import Image

cost_root = REPORT_DIR / "benchmark"
if cost_root.is_dir():
    display(pd.read_csv(cost_root / "selection_samples.csv"))
    display(pd.read_csv(cost_root / "research_cost.csv"))
    for figure_path in sorted(cost_root.glob("selection_quality_cost_*.png")):
        display(Image(filename=str(figure_path)))
else:
    display(
        Markdown(
            "Medición pendiente: ejecutar make benchmark y después make report para este estudio."
        )
    )

## Selección adaptativa con incertidumbre: extensión del mismo estudio

Los ajustes guardados de la familia ganadora forman un ensemble. Para cada candidato disponible se calcula su ganancia marginal de hipervolumen con cada miembro, usando solo respuestas ya reveladas. ensemble_mean maximiza la ganancia media y ensemble_ucb añade beta por su desviación muestral. La incertidumbre representa desacuerdo entre semillas; no es una probabilidad calibrada ni una garantía física. Los modelos permanecen congelados.

Esta extensión posterior al cierre conserva catálogos, inicios y presupuestos. La prueba ya fue examinada y sus comparaciones son exploratorias. Los costos de esta ejecución se presentan aparte del benchmark temporal repetido.

La evaluación puede distribuir bloques entre CPU workers sin cambiar las consultas. execution.json registra duración total; las tablas por trabajador separan trabajo acumulado y RSS individual. Los tiempos se observan bajo concurrencia y el RSS sumado puede contar páginas compartidas.


In [ ]:
adaptive_dir = REPORT_DIR / "adaptive"
if adaptive_dir.is_dir():
    display(pd.read_csv(adaptive_dir / "comparison.csv"))
    display(pd.read_csv(adaptive_dir / "paired_intervals.csv"))
    display(pd.read_csv(adaptive_dir / "uncertainty_diagnostics.csv"))
    display(pd.read_csv(adaptive_dir / "shared_costs.csv"))
    display(pd.read_csv(adaptive_dir / "decision_costs.csv"))
    if (adaptive_dir / "execution.json").is_file():
        display(pd.read_json(adaptive_dir / "execution.json", typ="series"))
    for filename in ("worker_jobs.csv", "worker_resources.csv"):
        if (adaptive_dir / filename).is_file():
            display(pd.read_csv(adaptive_dir / filename))
else:
    display(
        Markdown(
            "Extensión opcional pendiente: ejecutar `make adaptive` con el mismo ESTUDIO y reconstruir el informe con `make report`. No requiere reentrenar."
        )
    )

## Ablaciones y corrección durante el episodio

Se separan recompensa, representación, frecuencia de actualización e inicialización evolutiva. La corrección residual usa solamente consultas reveladas. La sensibilidad biobjetivo recalcula trayectorias triobjetivo congeladas: no son políticas reentrenadas para otra formulación.


In [ ]:
for filename in ["objective_sensitivity.csv"]:
    path = REPORT_DIR / filename
    if path.is_file():
        display(Markdown("**" + filename + "**"))
        display(pd.read_csv(path).head(30))

## Selección hacia generaciones posteriores

Los catálogos retenidos comparan las mismas consultas iniciales y candidatos. El control conserva tamaños por categoría y permite solapamiento de generaciones de forma explícita.

In [ ]:
path = REPORT_DIR / "chronological_selection_episodes.csv"
if path.is_file():
    grouped = pd.read_csv(path)
    display(
        grouped.groupby(
            ["experiment", "evaluation_seed", "split", "strategy", "budget"]
        )[["normalized_auc", "relative_hypervolume"]].mean()
    )

## Selección con TabPFN

Las once familias compiten usando validación. Ranking evalúa cada familia sobre los mismos catálogos; el predictor seleccionado alimenta greedy, RL y el adaptativo. La preparación del banco y la inferencia se distinguen del costo de decidir.


In [ ]:
ranking_path = REPORT_DIR / "ranking_runs.csv"
if ranking_path.is_file():
    ranking_table = pd.read_csv(ranking_path)
    display(ranking_table.loc[ranking_table.model.eq("tabpfn")])

## Sensibilidad al criterio de servicio
L/700 es el criterio principal adoptado. El caso sin límite Z reevalúa las mismas consultas: no corresponde a políticas reentrenadas para ese escenario. Los valores HV con referencias distintas no son directamente comparables.


In [ ]:
path = REPORT_DIR / "serviceability_sensitivity.csv"
if path.is_file():
    service = pd.read_csv(path)
    display(
        service.groupby(
            ["split", "serviceability", "strategy", "budget"], dropna=False
        )[["normalized_auc", "relative_hypervolume", "observed_failures"]].mean()
    )